In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import pmdarima as pm

In [2]:
graph1_1 = pd.read_csv('data/VAR_42631_56896-odecty.csv', sep=";")[['VAR_ID', 'Time of reading', 'Value']]
graph1_2 = pd.read_csv('data/VAR_42631_56896-odecty (1).csv', sep=";")[['VAR_ID', 'Time of reading', 'Value']]
graph2 = pd.read_csv('data/VAR_41494_55419-odecty.csv', sep=";")[['VAR_ID', 'Time of reading', 'Value']]
graph3 = pd.read_csv('data/VAR_97467_124449-odecty.csv', sep=";")[['VAR_ID', 'Time of reading', 'Value']]

In [3]:
def concatenate_datasets(df1, df2, output_file_path):
    try:
        concatenated_df = pd.concat([df1, df2], ignore_index=True)

        # Save the concatenated DataFrame to a new CSV file
        concatenated_df.to_csv(output_file_path, index=False)

        print(f"Successfully concatenated into '{output_file_path}'")
        print("\nInformation about the new concatenated dataset:")
        concatenated_df.info()

    except FileNotFoundError as e:
        print(f"Error: One of the files was not found. Please check the file paths.")
        print(f"Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


output_file = 'data/VAR_42631_56896-odecty-full.csv'

concatenate_datasets(graph1_1, graph1_2, output_file)
graph1 = pd.read_csv(output_file)

Successfully concatenated into 'data/VAR_42631_56896-odecty-full.csv'

Information about the new concatenated dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25258 entries, 0 to 25257
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   VAR_ID           25258 non-null  int64  
 1   Time of reading  25258 non-null  object 
 2   Value            25258 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 592.1+ KB


In [4]:
def check_periodicity(df, time_column):
    """
    Checks for periodicity in a DataFrame's time column and prints a message if a
    periodic timestamp is missed.

    Args:
        df (pd.DataFrame): The DataFrame to check.
        time_column (str): The name of the column containing the timestamps.
    """
    if time_column not in df.columns:
        print(f"Error: The column '{time_column}' was not found in the DataFrame.")
        return

    # Convert the time column to datetime objects
    df[time_column] = pd.to_datetime(df[time_column])

    # Sort the DataFrame by the time column to ensure correct calculation
    df = df.sort_values(by=time_column).reset_index(drop=True)

    # Calculate the time difference between consecutive rows
    time_diffs = df[time_column].diff()

    # Find the most common time difference, which represents the periodicity
    periodicity = time_diffs.mode()

    if periodicity.empty:
        print("Could not determine a consistent periodicity from the data.")
        return

    # Use the first element of the mode, as there could be multiple modes
    most_common_period = periodicity[0]
    print(f"\nDetermined a common periodicity of: {most_common_period}")

    # Find timestamps that do not match the most common period (with a small tolerance)
    missed_timestamps = time_diffs[time_diffs > most_common_period + pd.Timedelta(minutes=5)]
    missed_timestamps_num = 0
    
    if not missed_timestamps.empty:
        print("\nMissed periodic timestamp(s) detected:")
        for index, diff in missed_timestamps.items():
            prev_timestamp = df.loc[index - 1, time_column]
            current_timestamp = df.loc[index, time_column]
            missed_timestamps_num += 1
            # print(f"  - A gap of {diff} was found between {prev_timestamp} and {current_timestamp}.")
        print(f"\nMissed {missed_timestamps_num} timestamps out of {len(df)} measurements.")
    else:
        print("\nNo missed periodic timestamps were detected.")
        

check_periodicity(graph1, 'Time of reading')


Determined a common periodicity of: 0 days 00:20:00

Missed periodic timestamp(s) detected:

Missed 556 timestamps out of 25258 measurements.


In [5]:

def fill_missing_timestamps_with_arima(df, time_column, value_column):
    """
    Identifies and fills missing timestamps in a DataFrame using a fast ARIMA model.

    Args:
        df (pd.DataFrame): The DataFrame with time series data.
        time_column (str): The name of the timestamp column.
        value_column (str): The name of the value column to forecast.

    Returns:
        pd.DataFrame: A new DataFrame with missing timestamps filled.
    """
    print("\nAttempting to fill missing timestamps using an ARIMA model...")

    # Set the time column as the index
    df = df.set_index(time_column)
    
    # Resample to identify the missing timestamps. We use a high frequency to catch all gaps.
    freq = pd.infer_freq(df.index)
    if freq is None:
        print("Could not infer a consistent frequency from the data. Please check for gaps.")
        return df

    # Re-index the DataFrame to a continuous time series based on the inferred frequency
    # This will create NaNs for the missing timestamps
    full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
    df_resampled = df.reindex(full_index)

    missing_timestamps = df_resampled[df_resampled[value_column].isnull()]
    
    if missing_timestamps.empty:
        print("No missing timestamps found to fill.")
        return df.reset_index()

    print(f"Found {len(missing_timestamps)} missing timestamps. Using ARIMA to forecast values.")

    try:
        # Use auto_arima to automatically find the best ARIMA model
        model = pm.auto_arima(df[value_column], seasonal=False, stepwise=True,
                              suppress_warnings=True, error_action="ignore",
                              n_jobs=-1)
        
        # Forecast the missing values
        forecasted_values = model.predict(n_periods=len(missing_timestamps))
        
        # Create a new DataFrame for the forecasted values
        forecasted_df = pd.DataFrame({
            time_column: missing_timestamps.index,
            value_column: forecasted_values,
            'VAR_ID': df['VAR_ID'].iloc[0] # Assuming VAR_ID is constant for the series
        })
        
        # Merge the original and forecasted data
        filled_df = pd.concat([df.reset_index(), forecasted_df], ignore_index=True)
        filled_df = filled_df.sort_values(by=time_column).reset_index(drop=True)

        print("Successfully filled missing timestamps.")
        return filled_df
        
    except Exception as e:
        print(f"An error occurred during ARIMA forecasting: {e}")
        return df.reset_index()
    
filled_df = fill_missing_timestamps_with_arima(graph1, 'Time of reading', 'Value')



Attempting to fill missing timestamps using an ARIMA model...
Could not infer a consistent frequency from the data. Please check for gaps.
